# 根の探索

方程式 \(f(x)=0\) の解を数値的に求める。
計算結果は近似解なので、残差 \(|f(x)|\) と停止条件を確認する。


In [1]:
# 元Notebookの例
def f(x):
    return x**3 - x**2 + x - 1


In [2]:
from math import isfinite

def bisection(func, a, b, xtol=1e-8, ftol=1e-10, max_iter=100):
    if not (isfinite(a) and isfinite(b) and a < b):
        raise ValueError("a and b must be finite with a < b")
    if xtol <= 0 or ftol < 0 or max_iter < 1:
        raise ValueError("invalid tolerance or iteration limit")
    fa, fb = func(a), func(b)
    if not (isfinite(fa) and isfinite(fb)):
        raise ValueError("function must be finite at the endpoints")
    if abs(fa) <= ftol:
        return a
    if abs(fb) <= ftol:
        return b
    if (fa < 0) == (fb < 0):
        raise ValueError("endpoints must have opposite signs")
    for _ in range(max_iter):
        mid = a + (b - a) / 2
        fm = func(mid)
        if not isfinite(fm):
            raise ValueError("function must be finite in the interval")
        if abs(fm) <= ftol or b - a <= xtol:
            return mid
        if (fa < 0) != (fm < 0):
            b, fb = mid, fm
        else:
            a, fa = mid, fm
    raise RuntimeError("bisection did not converge")


In [3]:
root_bisect = bisection(f, -1.0, 2.0)
print(f"bisection: x={root_bisect:.10f}, |f(x)|={abs(f(root_bisect)):.2e}")
assert abs(root_bisect - 1.0) < 1e-8
assert abs(f(root_bisect)) < 1e-8

# 符号変化だけで区間を縮められる（この関数は単調ではない）。
nonmonotone_root = bisection(
    lambda x: (x + 1) * (x - 1) * (x - 2), -2.0, 3.0
)
assert abs(nonmonotone_root + 1.0) < 1e-8


bisection: x=1.0000000009, |f(x)|=1.86e-09


## Newton法

\(x_n\)のまわりで一次までTaylor展開し、接線が \(x\) 軸と交わる点を次の近似にする。
\[
f(x_n+\delta)\simeq f(x_n)+f'(x_n)\delta=0
\quad\Longrightarrow\quad
x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.
\]
初期値と導関数が必要で、収束しない場合もある。


In [4]:
def f_deriv(x):
    return 3 * x**2 - 2 * x + 1


In [5]:
def newton(func, deriv, x, xtol=1e-10, ftol=1e-10, max_iter=50):
    if xtol <= 0 or ftol < 0 or max_iter < 1:
        raise ValueError("invalid tolerance or iteration limit")
    for _ in range(max_iter):
        fx = func(x)
        if not isfinite(fx):
            raise ValueError("function must remain finite")
        if abs(fx) <= ftol:
            return x
        slope = deriv(x)
        if not isfinite(slope):
            raise ValueError("derivative must remain finite")
        if slope == 0:
            raise ZeroDivisionError("Newton derivative is zero")
        xnew = x - fx / slope
        if not isfinite(xnew):
            raise ValueError("Newton step is not finite")
        fnew = func(xnew)
        if not isfinite(fnew):
            raise ValueError("function must remain finite")
        if abs(fnew) <= ftol:
            return xnew
        if abs(xnew - x) <= xtol:
            raise RuntimeError("step is small but the residual is not")
        x = xnew
    raise RuntimeError("Newton method did not converge")


In [6]:
root_newton = newton(f, f_deriv, 3.0)
print(f"Newton: x={root_newton:.10f}, |f(x)|={abs(f(root_newton)):.2e}")
assert abs(root_newton - 1.0) < 1e-8
assert abs(f(root_newton)) < 1e-10

try:
    newton(lambda x: x**2 - 1, lambda x: 2*x, 0.0)
except ZeroDivisionError:
    print("zero derivative: stopped safely")
else:
    raise AssertionError("zero derivative should stop Newton's method")

assert newton(lambda x: x - 1, lambda x: 1.0, 0.0, max_iter=1) == 1.0


Newton: x=1.0000000000, |f(x)|=8.26e-14
zero derivative: stopped safely


## 準Newton法：割線法

導関数を2点の差分で近似し、Newton法と同じ接線の考え方を使う。
\[
x_{n+1}=x_n-f(x_n)
\frac{x_n-x_{n-1}}{f(x_n)-f(x_{n-1})}.
\]
解析的な導関数は不要だが、初期値を2つ与える。
分母が0に近い場合や不適切な初期値では失敗しうる。


In [7]:
def secant(func, x0, x1, xtol=1e-10, ftol=1e-10, max_iter=50):
    if xtol <= 0 or ftol < 0 or max_iter < 1:
        raise ValueError("invalid tolerance or iteration limit")
    f0, f1 = func(x0), func(x1)
    if not (isfinite(f0) and isfinite(f1)):
        raise ValueError("function must remain finite")
    for _ in range(max_iter):
        if abs(f1) <= ftol:
            return x1
        denominator = f1 - f0
        if denominator == 0:
            raise ZeroDivisionError("secant denominator is zero")
        x2 = x1 - f1 * (x1 - x0) / denominator
        if not isfinite(x2):
            raise ValueError("secant step is not finite")
        f2 = func(x2)
        if not isfinite(f2):
            raise ValueError("function must remain finite")
        if abs(f2) <= ftol:
            return x2
        if abs(x2 - x1) <= xtol:
            raise RuntimeError("step is small but the residual is not")
        x0, f0, x1, f1 = x1, f1, x2, f2
    raise RuntimeError("secant method did not converge")


In [8]:
root_secant = secant(f, 0.0, 2.0)
print(f"secant: x={root_secant:.10f}, |f(x)|={abs(f(root_secant)):.2e}")
assert abs(root_secant - 1.0) < 1e-8
assert abs(f(root_secant)) < 1e-10

try:
    secant(lambda x: x**2 + 1, -1.0, 1.0)
except ZeroDivisionError:
    print("zero secant denominator: stopped safely")
else:
    raise AssertionError("zero denominator should stop the secant method")

assert secant(lambda x: x - 1, 0.0, 2.0, max_iter=1) == 1.0


secant: x=1.0000000000, |f(x)|=2.07e-14
zero secant denominator: stopped safely


## 固定点問題

\(f(x)=0\)を \(x=g(x)\) と書き直し、
\[
x_{n+1}=g(x_n)
\]
を反復する。解 \(x_*\)の近くで \(|g'(x_*)|<1\)なら、反復は局所的に収束する。


In [9]:
# 元Notebookの例。f(x)=0をx=g(x)に書き直したもの。
def g(x):
    return -x**3 + x**2 + 1


In [10]:
def fixed_point(mapping, x, relaxation=1.0,
                xtol=1e-10, ftol=1e-10, max_iter=100):
    if not 0 < relaxation <= 1:
        raise ValueError("relaxation must be in (0, 1]")
    if xtol <= 0 or ftol < 0 or max_iter < 1:
        raise ValueError("invalid tolerance or iteration limit")
    for _ in range(max_iter):
        gx = mapping(x)
        if not isfinite(gx):
            raise ValueError("mapping must remain finite")
        xnew = x + relaxation * (gx - x)
        if not isfinite(xnew):
            raise ValueError("fixed-point step is not finite")
        residual = abs(mapping(xnew) - xnew)
        if abs(xnew - x) <= xtol:
            if residual <= ftol:
                return xnew
            raise RuntimeError("step is small but the fixed-point residual is not")
        x = xnew
    raise RuntimeError("fixed-point iteration did not converge")


In [11]:
root_fixed = fixed_point(g, 1.5, relaxation=0.5)
print(f"fixed point: x={root_fixed:.10f}, |g(x)-x|={abs(g(root_fixed)-root_fixed):.2e}")
assert abs(root_fixed - 1.0) < 1e-8
assert abs(g(root_fixed) - root_fixed) < 1e-10


fixed point: x=1.0000000000, |g(x)-x|=0.00e+00


## 講義資料の演習

次の関数を用いる：
\[
f(x)=x^3-x-2,\qquad
g(x)=\cos x-x,\qquad
h(x)=\log x+x-3.
\]


In [12]:
from math import cos, log

def f_task(x):
    return x**3 - x - 2

def g_task(x):
    return cos(x) - x

def h_task(x):
    return log(x) + x - 3


1. \(f,g,h\)をそれぞれ二分法で解く。\(h\)は対数の定義域 \(x>0\) で符号が変わる区間を選ぶ。
2. \(f,g,h\)をNewton法と割線法で解き、残差を確かめる。
3. \(g,h\)をそれぞれ固定点問題に書き直し、初期値と収束の様子を比べる。
